[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/landonking-gif/glasses3d/blob/main/colab/glasses3d_colab.ipynb)

# glasses3d on Colab

Reconstruct a 3D world from Meta Ray-Ban glasses footage using a Colab GPU.

**Set the runtime first:** Runtime -> Change runtime type -> GPU.

Two modes:

| Mode | What it needs | Works on |
|---|---|---|
| **Offline** (recommended) | a recorded clip | any GPU, including a free T4 |
| **Live** | phone streaming through a tunnel | L4 / A100 or better |

Offline is not the fallback — it is the better path. Recorded clips are 3K/60,
about **9x the pixels** of the 720p live stream, and there is no latency budget
to fight. Live is for interactivity, not fidelity.


## 1. What GPU did we get?

Colab allocates opportunistically — the same notebook gets a T4 one run
and an L4 the next — so capability has to be measured, not assumed.


In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or
      'No GPU. Runtime > Change runtime type > GPU, then rerun.')


## 2. Install

First run takes ~5-10 minutes. Mounting Drive caches the repo and the
model weights so later sessions skip most of it — worth doing, because a
free-tier session is capped at 12 hours and dies after 90 minutes idle.


In [ ]:
#@title Mount Drive (optional but recommended)
USE_DRIVE = True  #@param {type:'boolean'}

import os
ROOT = '/content'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/glasses3d-workspace'
    os.makedirs(ROOT, exist_ok=True)
print('workspace:', ROOT)


In [ ]:
#@title Get the code
REPO = os.path.join(ROOT, 'glasses3d')

GIT_URL = 'https://github.com/landonking-gif/glasses3d'  #@param {type:'string'}

if GIT_URL:
    if os.path.isdir(os.path.join(REPO, '.git')):
        !cd {REPO} && git pull --rebase
    else:
        !git clone {GIT_URL} {REPO}
elif not os.path.isdir(REPO):
    raise SystemExit('Set GIT_URL, or upload server/ into ' + REPO)

assert os.path.isdir(REPO), REPO
sys.path.insert(0, os.path.join(REPO, 'server'))
print('repo:', REPO)


In [ ]:
#@title Install dependencies (~5-10 min first run)
!pip -q install opencv-python-headless websockets

# MapAnything. The Apache-2.0 weights are the default in backends.py; the
# other variant is CC-BY-NC and research-only.
MA = os.path.join(ROOT, 'map-anything')
if not os.path.isdir(MA):
    !git clone -q https://github.com/facebookresearch/map-anything {MA}
!pip -q install -e {MA}
print('installed')


## 3. Capability and budget check

Tells you which mode is viable on the GPU you were assigned, and what the
run will cost, instead of letting you discover either one later.


In [ ]:
from backends import describe_gpu, estimate_cost, budget_advice

gpu = describe_gpu()
for k, v in gpu.items():
    print(f'{k:22} {v}')

if not gpu['available']:
    print('\nNo CUDA. Runtime > Change runtime type > GPU.')
elif gpu['live_viable']:
    print(f"\n{gpu['name']}: live mode should work. Offline still gives better quality.")
else:
    print(f"\n{gpu['name']}: OFFLINE ONLY.")
    print(f"Estimated {gpu['expected_slam_fps']} fps for live tracking; ~8 is the floor")
    print('for it to feel live. Use offline mode — better input, no latency budget.')

print('\n--- Colab Pro budget ---')
for line in budget_advice(gpu):
    print(' *', line)

for h in (0.25, 1.0, 4.0):
    c = estimate_cost(h, gpu)
    if 'error' not in c:
        print(f"  {h:>4}h = {c['compute_units']:>6.2f} CU "
              f"({c['percent_of_monthly_pro']}% of a 100 CU month)")


### The two-phase workflow (Colab Pro)

Pro includes **100 compute units a month**. That is ~84 hours of T4 but only
**~18 hours of A100 40GB** — the premium GPUs are a budget to spend
deliberately, not a default to leave running.

| GPU | CU/hr | Hours on a full allowance |
|---|---|---|
| T4 | 1.19 | ~84 |
| L4 | ~2.6 | ~38 |
| A100 40GB | 5.40 | ~18.5 |
| A100 80GB | 7.52 | ~13 |

So split the work:

**Phase 1 — on a T4.** Install, download weights, calibrate, and do a
`--backend mock` run to prove the pipeline end to end. Cache everything to
Drive. None of this needs an A100, and the ~10 minute first install costs
~0.9 CU on an A100 versus ~0.2 on a T4 — every single session.

**Phase 2 — switch to A100.** Runtime -> Change runtime type -> A100
(the High-RAM slider picks 40GB vs 80GB). Rerun the setup cells; they hit
the Drive cache and finish in seconds. Then reconstruct, download
`scene.ply`, and **disconnect immediately** — idle premium sessions bill at
the same rate as busy ones.

Done this way, a reconstruction costs well under 1 CU and the allowance
lasts. Done carelessly, an afternoon of leaving an A100 connected is a
quarter of the month.


### Phase 1 check: prove the pipeline before spending A100 time

Runs the test suite and a full mock reconstruction — every stage except
the neural model itself. If this passes, the only untested thing left is
MapAnything, which is exactly what Phase 2 is for. Costs almost nothing
on a T4, and finding a broken path here rather than on an A100 is the
entire point of splitting the work.


In [ ]:
#@title Verify everything except the model
import subprocess, os

ok = True
for t in ['tracker', 'worldmodel', 'perception', 'export', 'reconstruct']:
    r = subprocess.run(['python3', f'{REPO}/tools/test_{t}.py'],
                       capture_output=True, text=True)
    tail = [l for l in r.stdout.strip().split('\n') if 'passed' in l]
    print(f"  {t:12} {tail[-1] if tail else 'no result'}")
    ok &= (r.returncode == 0)

# A synthetic clip, so this works before you have uploaded any footage.
import cv2, numpy as np
demo = os.path.join(ROOT, 'synthetic.mp4')
vw = cv2.VideoWriter(demo, cv2.VideoWriter_fourcc(*'mp4v'), 24, (640, 480))
rng = np.random.RandomState(0)
tex = cv2.GaussianBlur(rng.randint(0, 255, (700, 900, 3)).astype(np.uint8), (7, 7), 2)
for i in range(72):
    vw.write(tex[100 + int(2 * i):100 + int(2 * i) + 480, 100 + i * 3:100 + i * 3 + 640])
vw.release()

r = subprocess.run(['python3', f'{REPO}/server/reconstruct.py',
                    '--video', demo, '--out', os.path.join(ROOT, 'mockout'),
                    '--backend', 'mock', '--views', '8'],
                   capture_output=True, text=True)
print('\n' + r.stdout.strip()[-500:])
ok &= (r.returncode == 0)
print('\n' + ('Phase 1 PASSED — safe to switch to an A100.' if ok else
              'Phase 1 FAILED — fix this on the cheap GPU, not the expensive one.'))


---
# Offline mode

Upload a clip, get `scene.ply` back.


### 3a. Calibration — OPTIONAL, and no printer needed

The glasses use a 12MP ultrawide with strong barrel distortion. Correcting
it measurably improves geometry, but **you can skip this entirely** —
MapAnything estimates intrinsics itself. Skipping costs accuracy, not
function.

| | Effort | Result |
|---|---|---|
| **Skip** | none | MapAnything infers intrinsics. Works. Slightly softer geometry, less reliable metric scale |
| **Screen** | ~2 min, once ever | Near-printed quality, no printer |
| **Printed** | needs a printer | Marginally best — flat rigid target, no glare |

#### The screen method

1. Open `calib/checkerboard-9x6.png` from this repo **fullscreen** on a
   laptop, tablet, or second monitor. A phone screen is usable but small —
   bigger is better.
2. Turn screen brightness **down** to about half. A blown-out white
   destroys corner contrast, and glare is the main way this method fails.
3. Wearing the glasses, record ~30s while moving *yourself* around the
   screen — closer, further, off to each side, tilted up and down. Get the
   board into the **frame corners**, where ultrawide distortion actually
   carries information. A board that stays centred yields coefficients
   that fit nothing.
4. Move slowly. Blurred frames are rejected and you want 20+ usable views.

**You never have to measure anything.** Intrinsics are invariant to the
target's physical size — scaling the board only scales the recovered
camera *positions*, which this pipeline discards. Verified directly:
assuming 25 mm and 137 mm squares yields bit-identical `fx`, `cx` and
distortion. So `--square-mm` can stay at its default and the on-screen
size is irrelevant.

Do this once, ever — intrinsics belong to the glasses, not the scene.


In [ ]:
#@title Calibrate (or press Cancel to skip)
from google.colab import files

os.makedirs(os.path.join(REPO, 'calib'), exist_ok=True)
print('Upload the checkerboard video — or press Cancel to skip calibration.')
up = files.upload()

if up:
    board = os.path.join(REPO, 'calib', list(up)[0])
    open(board, 'wb').write(list(up.values())[0])
    # --square-mm is arbitrary: it does not affect intrinsics, only the
    # extrinsic translations, which are discarded.
    !cd {REPO} && python3 server/calibrate.py --video "{board}" --square-mm 25
    print('\nWant RMS < 0.5 px. Higher means retry with slower motion, less glare,',
          'and the board pushed further into the frame corners.')
else:
    print('Skipped — MapAnything will estimate intrinsics from the footage.')
    print('Reconstruction still works; geometry is just softer and metric scale')
    print('less reliable. You can calibrate later and re-run without redoing anything.')


### 3b. Reconstruct


In [ ]:
#@title Upload a clip
from google.colab import files

print('Upload your glasses clip:')
up = files.upload()
assert up, 'no file uploaded'

clip = os.path.join(ROOT, list(up)[0])
open(clip, 'wb').write(list(up.values())[0])
print('%s (%.1f MB)' % (clip, os.path.getsize(clip) / 1e6))


In [ ]:
#@title Run reconstruction
# VIEWS drives both quality and VRAM, so the sensible default depends on the
# card you were actually given. Override AUTO_TUNE to set it yourself.
AUTO_TUNE = True  #@param {type:'boolean'}
VIEWS = 32        #@param {type:'integer'}
WIDTH = 518       #@param {type:'integer'}
MIN_CONF = 0.5    #@param {type:'number'}
MAX_POINTS = 1500000  #@param {type:'integer'}

if AUTO_TUNE and gpu.get('available'):
    vram = gpu['vram_gb']
    #  ~4 GB of that is MapAnything's 1B fp32 weights; the rest is activations,
    #  which scale with view count.
    VIEWS = 24 if vram < 20 else 48 if vram < 45 else 96
    WIDTH = 518 if vram < 20 else 700
    print(f"auto-tuned for {gpu['name']} ({vram} GB): VIEWS={VIEWS} WIDTH={WIDTH}")

OUT = os.path.join(ROOT, 'out')
cmd = (f'cd {REPO} && python3 server/reconstruct.py'
       f' --video "{clip}" --out "{OUT}" --backend mapanything'
       f' --views {VIEWS} --width {WIDTH}'
       f' --min-conf {MIN_CONF} --max-points {MAX_POINTS}')
print(cmd)
!{cmd}

# If this OOMs, halve VIEWS before changing anything else — it is the
# dominant term in activation memory.


In [ ]:
#@title Inspect and download scene.ply
import json
meta = json.load(open(os.path.join(OUT, 'meta.json')))
for k, v in meta.items():
    print(f'{k:16} {v}')

from google.colab import files
files.download(os.path.join(OUT, 'scene.ply'))


### 3c. Where scene.ply goes

| Target | How |
|---|---|
| Web | drag onto [superspl.at/editor](https://superspl.at/editor) — do this first, fastest feedback |
| Blender | 3DGS Render addon (KIRI) -> Import PLY |
| Unity | [UnityGaussianSplatting](https://github.com/aras-p/UnityGaussianSplatting) |
| Unreal | [SplatRenderer](https://github.com/DazaiStudio/SplatRenderer-UEPlugin), UE 5.5+ |
| Quest VR | via Unity — keep under ~400k splats for 72fps |


---
# Live mode

On Colab Pro this is actually reachable — but **only on an A100**. An L4
lands near 4.5 fps against the ~8 fps floor for it to feel live; an A100
lands near 9.8. The capability check above is the arbiter.

**The structural problem:** Colab has no public inbound address, so the
phone cannot reach it directly. A tunnel is mandatory, and it adds
100-300 ms on top of the glasses -> phone hop — well above the 200 ms
a local GPU box would hit. The tunnel URL also changes on every restart,
so the phone needs reconfiguring each session.

**And it is the most expensive thing you can do with the allowance.** A100
live streaming burns 5.40 CU/hr against 100 CU/month, so a one-hour session
is 5% of the month. Get the offline path working first — it is higher
quality and costs a fraction as much.


In [ ]:
#@title Open a public tunnel
import re, time, subprocess

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

PORT = 8765
log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=log, stderr=subprocess.STDOUT)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        url = m.group(0)
        break

if url:
    print('Tunnel up.')
    print('Set serverURL in Glasses3DRelay.swift to:')
    print('   ' + url.replace('https://', 'wss://'))
else:
    print('Tunnel did not come up — check /content/tunnel.log')


In [ ]:
#@title Run the live pipeline
# ingest.py only *receives* frames - it renders them and stops there. The live
# loop is server/live.py, which tracks the camera, gates keyframes, densifies
# on a worker thread and writes scene.ply/points.ply/scene.json as it goes.

DETECT = 'pencil,mug,book'  #@param {type:'string'}
OUT_LIVE = os.path.join(ROOT, 'live-out')

# --track mast3r and --backend mapanything are the real models; both need CUDA.
# Swap either to 'mock' to prove the path without burning compute units.
cmd = (f'cd {REPO} && python3 -u server/live.py'
       f" --source ws --port 8765 --backend mapanything --track mast3r"
       f" --detect '{DETECT}' --out '{OUT_LIVE}'")
print(cmd)
print('\\nArtifacts land in', OUT_LIVE, '- download points.ply any time and')
print('drop it on the walkthrough viewer; it updates while this runs.\\n')
!{cmd}


### Live-mode reality check

- **Free-tier T4 will not keep up.** Roughly 2-3 fps for dense SLAM against
  the ~8 fps floor for it to feel live.
- **Sessions die.** 90 minutes idle, 12 hours absolute on free tier. The
  tunnel URL changes every restart, so the phone needs reconfiguring.
- **Glasses battery** is the other clock. Design for 2-5 minute captures.

If live is the actual goal rather than a demo, a persistent GPU box
(RunPod, Lambda) with a stable address is a much better fit than Colab.
Colab is excellent at the offline path and awkward at this one.
